# Provision a Vertex AI Training Cluster (with hardware fallback)

## Objective
Stand up a VTC cluster end-to-end the way the runbook does (PDF Steps 1–3): set
parameters, enable APIs, wire Private Services Access, create the Filestore
`/home` and a code-transfer bucket, then create the cluster — walking a
**hardware fallback chain** so a stockout on your first-pick accelerator doesn't
stop the run.

This is the **Notebooks path** — the hands-on twin of the Terraform path. Both
read the same contract (`configs/cluster_config.yaml`) and call the same
`scripts/create_cluster.sh`, so they never drift. You do **not** need to run
Terraform first.

A VTC cluster is a `modelDevelopmentClusters` resource on the v1beta1 Vertex AI
API — **Preview**, typically allowlisted. TPU profiles ride the GKE orchestrator
(also Preview); GPU/CPU profiles ride Slurm.

Docs: [VTC overview](https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/training/training-clusters/overview) ·
[Get started](https://docs.cloud.google.com/gemini-enterprise-agent-platform/machine-learning/training/training-clusters/get-started)

## 1. Set parameters (`@param`)

Edit these in the form, then the next cell **regenerates the contract**
(`configs/cluster_config.yaml`) from them. Keeping the knobs as `@param` and
writing the file from one place is how the notebook stays the source of truth the
Terraform path also reads.

In [1]:
# @markdown ### What to run this round
# @markdown - **cpu** — plumbing validation: one CPU cluster.
# @markdown - **gpu-h100** — real GPU cluster on **A3 / H100** (8× H100, us-east4 — your granted quota).
# @markdown - **full-chain** — fallback demo: a4b200 (no B200 quota) → a3h100 (H100 ✓) → cpu.
# @markdown Runtime knob only (mirrors `create_cluster.sh --profiles` / Terraform `var.profiles`); the config holds the full `fallback_chain`.
MODE = "full-chain"  # @param ["cpu", "gpu-h100", "full-chain"]
PROFILES = {"cpu": "cpu", "gpu-h100": "a3h100", "full-chain": ""}[MODE]
print("mode:", MODE, "| profiles:", PROFILES or "(full fallback_chain)")

mode: full-chain | profiles: (full fallback_chain)


In [2]:
import os, subprocess

# @markdown ### Project + location  (leave PROJECT blank to use your current gcloud project)
PROJECT = ""                 # @param {type:"string"}
if not PROJECT:
    PROJECT = subprocess.run(
        ["gcloud", "config", "get-value", "project"],
        capture_output=True, text=True,
    ).stdout.strip()
REGION  = "us-east4"         # @param {type:"string"}
ZONE    = "us-east4-a"       # @param {type:"string"}

# @markdown ### Network (must already exist; needs a subnet in REGION)
NETWORK = "beusebio-network" # @param {type:"string"}
SUBNET  = "beusebio-network" # @param {type:"string"}

# @markdown ### Filestore (shared /home — created in ZONE; us-central1 one won't work for us-east4)
FILESTORE_ID = "filestore-test"  # @param {type:"string"}
SHARE_NAME   = "vol1"            # @param {type:"string"}
TIER         = "BASIC_HDD"       # @param ["BASIC_HDD","BASIC_SSD","ZONAL","REGIONAL"]
CAPACITY     = "1TB"             # @param {type:"string"}

# @markdown ### Code-transfer bucket  (blank -> <project>-vtc-temp)
BUCKET = ""                  # @param {type:"string"}
BUCKET = BUCKET or f"{PROJECT}-vtc-temp"

# @markdown ### Workload (Gemma 4)
GEMMA_MODEL = "gemma4-4b"    # @param {type:"string"}
DATA_DIR    = "/gcs/your-bucket/dataset"  # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT
print("project:", PROJECT, "| region:", REGION, "| zone:", ZONE, "| bucket:", BUCKET)

project: sandbox-401718 | region: us-east4 | zone: us-east4-a | bucket: sandbox-401718-vtc-temp


## 2. Edit the fallback chain, then write the contract

`fallback_chain` is the ordered list of hardware the create walks **top-to-bottom**,
stopping at the first profile that provisions. Put your scarcest accelerator
first; end with `cpu` so a run can always validate networking + Slurm. Edit the
Python list below — each entry maps 1:1 to a profile in the YAML.

> `%%writefile` writes its cell *literally* and can't see Python variables, so we
> build the YAML from the `@param` values above with an f-string and write it.
> (If you'd rather hand-edit YAML directly, a pure `%%writefile ../configs/cluster_config.yaml`
> cell with the full literal works too — just keep the scalars in sync.)

In [ ]:
# Ordered hardware fallback chain. Reorder / trim to taste.
# VTC GPU machine types: A3 = a3-megagpu-8g (a3-highgpu / a2 are "not allowed for VM"),
# A4 = a4-highgpu-8g. GPU pools: render sets provisioning_model=ON_DEMAND (shared capacity)
# and boot_disk_type=hyperdisk-balanced. a3-megagpu needs NVIDIA_H100_MEGA quota.
# To GUARANTEE capacity instead of on-demand, add `reservation: <name>` to a GPU profile
# (forces RESERVATION + reservation_affinity) and create it via notebooks/00-reserve-gpu.ipynb.
FALLBACK_CHAIN = [
    # name      accel  orchestrator  machine_type        accel_type/count   nodes
    {"name": "a4b200", "accelerator": "gpu", "orchestrator": "slurm",
     "machine_type": "a4-highgpu-8g", "accelerator_type": "NVIDIA_B200", "accelerator_count": 8,
     "node_count": 2},
    {"name": "a3h100", "accelerator": "gpu", "orchestrator": "slurm",
     "machine_type": "a3-megagpu-8g", "accelerator_type": "NVIDIA_H100_MEGA_80GB", "accelerator_count": 8,
     "node_count": 1},
    # TPU is DISABLED: this v1beta1 API rejects tpu_type / gke_spec ("Cannot find field").
    {"name": "cpu", "accelerator": "cpu", "orchestrator": "slurm",
     "machine_type": "n2-standard-16", "node_count": 2},
]

import yaml, pathlib
config = {
    "project": {"id": PROJECT, "region": REGION, "zone": ZONE},
    "network": {"vpc": NETWORK, "subnet": SUBNET, "psa_prefix_length": 20},
    "filestore": {"instance_id": FILESTORE_ID, "share_name": SHARE_NAME, "tier": TIER, "capacity": CAPACITY},
    "bucket": {"name": "" if BUCKET == f"{PROJECT}-vtc-temp" else BUCKET},
    "login_node": {"machine_type": "n2-standard-8", "boot_disk_type": "pd-standard", "boot_disk_size_gb": 100},
    "fallback_chain": FALLBACK_CHAIN,
    "workload": {
        "model": GEMMA_MODEL, "task": "pretrain",
        "gpu": {"framework": "nemo", "recipe": f"recipes/pretrain/{GEMMA_MODEL.replace('-','_')}_pt.py", "slurm_type": "hcc-a4"},
        "tpu": {"framework": "maxtext", "recipe": f"MaxText/configs/{GEMMA_MODEL}.yml"},
        "data_dir": DATA_DIR,
    },
}
CONFIG_PATH = pathlib.Path("../configs/cluster_config.yaml")
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
print(f"wrote {CONFIG_PATH.resolve()}")
print(CONFIG_PATH.read_text())

## 3. Authenticate + enable APIs

In [4]:
!gcloud config set project {PROJECT}
!gcloud config set compute/region {REGION}
# NOTE: the Filestore API's service name is `file.googleapis.com` (NOT filestore.googleapis.com).
# In sandbox-401718 these are ALL already enabled, so this is a no-op (safe to re-run or skip).
!gcloud services enable aiplatform.googleapis.com file.googleapis.com \
    compute.googleapis.com servicenetworking.googleapis.com storage.googleapis.com \
    || echo "(skip: APIs already enabled — fine to continue)"

Updated property [core/project].
Updated property [compute/region].
Operation "operations/acat.p2-757654702990-83c52815-3929-4456-b6b2-32a86a76546d" finished successfully.


## 4. Private Services Access 

Filestore attaches to your VPC over PSA. This is idempotent — it only creates the
peering if it isn't already there.

In [5]:
import subprocess
peerings = subprocess.run(
    ["gcloud", "services", "vpc-peerings", "list", "--network", NETWORK,
     "--project", PROJECT, "--format", "value(peering)"],
    capture_output=True, text=True).stdout
if "servicenetworking-googleapis-com" in peerings:
    print("PSA connection already exists.")
else:
    print("Creating PSA connection...")
    subprocess.run(
        ["gcloud", "compute", "addresses", "create", f"google-managed-services-{NETWORK}",
         "--global", "--purpose=VPC_PEERING", "--prefix-length=20",
         f"--network={NETWORK}", f"--project={PROJECT}"], check=True)
    subprocess.run(
        ["gcloud", "services", "vpc-peerings", "connect",
         "--service=servicenetworking.googleapis.com", f"--network={NETWORK}",
         f"--ranges=google-managed-services-{NETWORK}", f"--project={PROJECT}"], check=True)

PSA connection already exists.


## 5. Filestore `/home` + code-transfer bucket

Filestore creation takes a few minutes — wait for it to finish before creating
the cluster, since the cluster references it as `home_directory_storage`.

In [6]:
# Filestore (idempotent-ish: ignores "already exists")
!gcloud filestore instances create {FILESTORE_ID} \
    --project={PROJECT} --zone={ZONE} --tier={TIER} \
    --file-share=name={SHARE_NAME},capacity={CAPACITY} \
    --network=name="projects/{PROJECT}/global/networks/{NETWORK}",connect-mode=PRIVATE_SERVICE_ACCESS \
    || echo "(continuing — instance may already exist)"

# Code-transfer bucket
!gcloud storage buckets create gs://{BUCKET} --project={PROJECT} --location={REGION} \
    --uniform-bucket-level-access || echo "(continuing — bucket may already exist)"

Waiting for [operation-1782423698739-6551adb6712c9-797010f0-ce28fb74] to finish
...done.                                                                       
Creating gs://sandbox-401718-vtc-temp/...
ERROR: (gcloud.storage.buckets.create) HTTPError 409: The requested bucket name is not available. The bucket namespace is shared by all users of the system. Please select a different name and try again.
(continuing — bucket may already exist)


## 6. Create the cluster — walk the fallback chain

`create_cluster.sh` renders the request JSON for each profile, POSTs the v1beta1
create, and polls the long-running operation. The **first** profile that comes up
wins; failures (stockout / quota / not allowlisted) fall through to the next.

Once a cluster is up, in-flight hardware faults are VTC's own job — it detects,
remediates (restart / reimage / replace), and resumes from checkpoint. The
fallback here is only for *creation-time* capacity.

> Tip: start with `--profiles cpu` to validate plumbing cheaply, or `--dry-run`
> to print the request bodies without creating anything.

In [7]:
# Dry run first (prints the JSON it WOULD POST, per profile — no resources created):
!bash ../scripts/create_cluster.sh --config ../configs/cluster_config.yaml --dry-run

==> project=sandbox-401718 region=us-east4 cluster_id=vtcgemma44
==> fallback chain: a4b200 a3h100 cpu
wrote /tmp/vtc_a4b200.5sKP.json
----------------------------------------------------------------------
==> trying profile 'a4b200'
    [dry-run] would POST:
      {
        "display_name": "vtc-gemma4-4b-a4b200",
        "network": {
          "network": "projects/sandbox-401718/global/networks/beusebio-network",
          "subnetwork": "projects/sandbox-401718/regions/us-east4/subnetworks/beusebio-network"
        },
        "node_pools": [
          {
            "id": "login",
            "machine_spec": {
              "machine_type": "n2-standard-8"
            },
            "scaling_spec": {
              "min_node_count": 1,
              "max_node_count": 1
            },
            "enable_public_ips": true,
            "zone": "us-east4-a",
            "boot_disk": {
              "boot_disk_type": "pd-standard",
              "boot_disk_size_gb": 100
            }
       

In [8]:
# Real create. Profiles come from the PROFILES cell at the top (runtime subset of the
# fallback_chain). Mirrors create_cluster.sh --profiles / terraform var.profiles.
# Winning id+profile -> .vtc_cluster_state (notebooks 02/03 read it back).
prof_flag = f"--profiles {PROFILES}" if PROFILES else ""   # empty PROFILES = full chain
!bash ../scripts/create_cluster.sh \
    --config ../configs/cluster_config.yaml \
    --cluster-id vtcgemma {prof_flag} \
    --state-out .vtc_cluster_state

==> project=sandbox-401718 region=us-east4 cluster_id=vtcgemma
==> fallback chain: a4b200 a3h100 cpu
wrote /tmp/vtc_a4b200.QAqS.json
----------------------------------------------------------------------
==> trying profile 'a4b200'
    create call rejected for 'a4b200':
      {
        "error": {
          "code": 400,
          "message": "List of found errors:\t1.Field: model_development_cluster.node_pools[1].boot_disk.boot_disk_type; Message: Machine type a4-highgpu-8g requires boot_disk_type to be unset (the default) or \"hyperdisk-balanced\"; got \"pd-standard\". See b/503144983.\t",
          "status": "INVALID_ARGUMENT",
          "details": [
            {
              "@type": "type.googleapis.com/google.rpc.BadRequest",
              "fieldViolations": [
                {
                  "field": "model_development_cluster.node_pools[1].boot_disk.boot_disk_type",
                  "description": "Machine type a4-highgpu-8g requires boot_disk_type to be unset (the default) 

In [22]:
# What won?
!cat .vtc_cluster_state 2>/dev/null || echo "no cluster yet"

CLUSTER_ID=vtcgemma
PROFILE=cpu


## Next
- **`02-run-training.ipynb`** — SSH in, validate Slurm, and launch Gemma on the
  winning hardware.
- **`03-teardown.ipynb`** — delete the cluster, Filestore, and bucket.